clustering :
the goal is to group the similar items together into clusters, used for data analysis, customer segmentation, recommender systems, search engines, image segmentation, semi supervised learning, dimensionality reduction and more

anomaly detection :
the goal is to learn the normal data and detect whenever an unusual one enters the pipeline, these abnormal ones are called anomalies or outliers and the normal ones are called inliers 
used in fraud detection, defective products and identifying new trends in the time series or removing outliers from a dataset before training another model 

density estimation :
this is the task of estimitating the probability density function of the random process that generated the dataset, density estimation is commonly used for anamoly detection as whenever an instance falls in very low density regions it is said to be an anomaly 

CLUSTERING Algorithms 

clustering is just like classification but an unsupervised version without labels 
see fig 9-1
it shows iris dataset on the left side it was represented using different markers it is a labeled so we can apply classification algorithms,
but on the right it is without the labels so we cannot use those algo so we use clustering algorithms 
as you can see we can esily see the lower cluster here but the other two are not recognisable, the dataset has two seperate features sepal length and width that are not represented here and clustering algos can make good use of all features 

clustering is used in a wide variety of tasks :

1. customer segemntation :
people who behave alike get grouped together easier personalisation 

2. data analysis :
it gives structure to any new dataset so we can explore it meaningfully

3. dimensionality reduction :
after clustering we calculate the affinity score which is a score that tells how much a data point belongs to a cluster 
so after this all the multidimensional features are of no use only the affinity scores matter
original feature = 10000
clusters = 20
new features = 20 dimensionality affnity vector 

4. feature engineering :
these affinity scores can be added as new features, like the geographical clustering we did on the california housing dataset in unit 2

5. anomaly detection :
if a data point is far from all the clusters its suspicious 

6. semi supervised learning :
when all the data dont have labels
cluster the data
assign each cluster the label shared by the few labeleld points inside it 
propogate those labels to all points in that cluster 
this increases your labeled data for training

7. search engines (image similarity search)

8. image segmentation :
each pixel is assigned to a cluster 
then we replace the pixel colour to clusters mean colour 
highlights the object boundaries 
used in object detection, tracking, medical imaging, comp vis tasks


so there is no universal definition of what clusters are they can be of different forms and shapes :
1. some algos will be looking for instances centered around a particular point called centroid 
2. will look for continuous regions of densely packed instances
3. some algos are hiearchial they will look for clusters inside the clusters 

we will look at 2 famous ones kmeans and dbscan and explore some of their applications such as non linear dimensionality reduction, semi supervised learning and anomaly detection 

K MEANS :

consider the unlabeled dataset in fig 9-2 the k means can cluster these dataset in few iterations only 

lets train a k means on this the algorithm will group the clusters by finding blob center and then assign each instance to the closest blob :

In [5]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

x, y = make_blobs(
    n_samples=2000,
    centers=[(-3, 3), (-3, 2), (-3, 1), (-1, 2.2), (0.3, 2)],
    cluster_std=[0.15, 0.15, 0.15, 0.40, 0.50],
    random_state=42
)

k = 5
kmeans = KMeans(n_clusters=k, random_state=42)
y_pred = kmeans.fit_predict(x)

you have to tell the no of clusters you want to specify, in this example we have set it to 5 but in general its not that easy 
the instance label is the index of the cluster it was assigned to this is not like the class labels in classification which are used as targets 

we can see the cluster assigned for each instance :

In [6]:
y_pred 

array([3, 2, 0, ..., 2, 3, 4])

this is stored as labels in kmeans class

In [7]:
kmeans.labels_

array([3, 2, 0, ..., 2, 3, 4])

we can also see the 5 centroids the algorithm found :

In [8]:
kmeans.cluster_centers_

array([[-2.97227212,  2.01040302],
       [ 0.43353751,  1.97095278],
       [-3.00408131,  3.00171197],
       [-0.95331001,  2.19762791],
       [-3.00482979,  1.01294344]])

we can also assign new instances to see which cluster they belongs to 

In [9]:
import numpy as np 
x_new = np.array([[0,2], [3,2], [-3,3], [-3,2.5]])
kmeans.predict(x_new)

array([1, 1, 2, 0])

if we plot the cluster decision boundaries we get a voronoi tessellation see get fig 9-3 
each centroid is represented by x

most of the instances are correctly assigned but you can see some instances between the top left and the center are not properly assigned, the kmeans doesnot behave very well when the blobs have different diameters because it only looks at the distance between the instances and the centre before assigning a cluster to the instance 

now instead of assigning each instance to a single cluster which is hard clustering we will be giving each instance a score per cluster which is called soft clustering
the score can be the distance between the instance and the cluster centroids or a similarity score (affinity score) such as the gaussian radial basis function we used in the 2nd chap 

in the kmeans class we can use the transform class to calculate the distance of each instance to every centroid  :

In [10]:
kmeans.transform(x_new).round(2)

array([[2.97, 0.43, 3.17, 0.97, 3.16],
       [5.97, 2.57, 6.09, 3.96, 6.09],
       [0.99, 3.58, 0.  , 2.2 , 1.99],
       [0.49, 3.47, 0.5 , 2.07, 1.49]])

if you have a high dimensional dataset this can be a good non linear dimensionality reduction technique, and we can also use these distances as extra features to train our model 


algorithm :

suppose u are given a centroid then u can easily label the instances by assigning them clusters by measuring which center is closer to them 

and if you are given the labels you can find the cluster centroids by calculating the mean of the instances in that cluster 

but if you dont have centroids and even labels then :
1. we pick the random k instances 
2. label them as one cluster 
3. update the centroid 
keep following this until the centroids stop moving, the algorithm is guaranteed to converge coz the mean squared distance between the instances and their closest centroids can only go down at each step and since it cannot be negative it will only go down eventually

see fig 9-4 :
1. the algorithm starts by placing the k centroids randomly (top left)
2. each point is assigned to the nearest centroid (top right)
3. now we move the centroids to the mean of the points assigned to them (middle left )
4. reassign the points (middle right)
5. update the centroids again (bottom left)
6. reassign the points (bottom right)


the time complexity is generally linear for this with regard to no of inastances m, the no of clusters k and no of dimensios but this is only true when the data is having a cluster architectue otherwise the time comp will increase exponentially 
this rarely happens and k means is the fastest algorithm 


look at fig 4-5 :
both the sol are wrong they are the result of different random initialization of centroids 
although the algorithm meant to be converge its not a guarantee it will always give the best possible clustering 
the algorithm heavily depends on the initial placing of the centroids if they are placed poorly(eg: two are placed inside the same blob) the clusters will merge incorrectly and the algorithm will converge at the local minima 


lets see how can we solve this :

centroid initialization methods :

1. sometimes you know where the cluster center should be maybe by running any another clustering algo, by visual inspection or by domain knowledge so you manually provide good initial centroids instead of k means guessing it randomly :

In [11]:
good_init = np.array([[-3,3], [-3,2], [-3,1], [-1,2], [0,2]])
kmeans = KMeans(n_clusters=5, init=good_init, n_init=1, random_state=42)
kmeans.fit(x)

KMeans(init=array([[-3,  3],
       [-3,  2],
       [-3,  1],
       [-1,  2],
       [ 0,  2]]),
       n_clusters=5, n_init=1, random_state=42)

n_init = 1 says to not run random initializations 

another solution is to run the algorithm multiple times and then select the best outcome, the no of times it runs depends on n_init hyperparam by default it is 10 when you call fit(), 
the sk learn knows which sol is best by calculating a performance metric called model's inertia, which is the sum of the squared distances between the instances and their closest centroids, which is 
219.4 (left fig 9-5)
258.6 (right fig 9-5)
211.6 for the model in fig 9-3

so the model in fig 9-3 will be selected 
we can access this inertia param by :

In [12]:
kmeans.inertia_

357.04295213609686

the score method returns the negative inertia coz according to the predictors logic the greater the value its better 
but inertia cant be greater so its represented in negative so even if you represent it in greater it will be less only weird logic ik 

In [13]:
kmeans.score(x)

-357.0429521360969

in 2006 a paper was published that tells how can the initialization step tends to select the centroids at a distant from each other and this approach ensures it does not get stuck into the suboptimal sol 
we need some extra computation for this but def worth it as it drastically decreases the no of times the algo runs again and again to find the optimal sol 

1. pick the first centroid randomly c^(1)
2. now for each point calculate the dist between that point and the closest centroid already chosen copy 8, square this distance copy 9, now choose the next centroid with probab copy 10
now points far from existing centroids will have large d^2 so it will have high chance of being chosen 
and if the point is close to centroid, it will have low d^2 so very low chance
this avoids two centres starting inside the same blobs 
3. repeat until k centroids are chosen 

the k means class uses this initialization method by default 



Accelerated k-means and mini batch k-means :

another approach was given in 2003 paper called elkans acceleated k means (2003) which reduces the no of distance computations which is the most expensive part of k means 
he used triangle inequality (a straight line is always the shortest distance between the two points )
it keeps track of upper bounds on how far a point might be from its assigned centroid 
lower bounds on how close it could be to other centroids 
using this the algorithm can often prove point x cannot be switched to centroid c2
it is good when the clusters are well seperated , dimensionality is not too high, the no of clusters k is large you can use this by setting algorithm = "elkan"

another important variant was proposed in 2010 which says instead of using full dataset at each iteration we can use mini batches moving the centroids slightly at each iteration, this speeds up the algorithm and makes it possible to cluster huge datasets that do not fit in memory,
the sklearn implements this by :

In [14]:
from sklearn.cluster import MiniBatchKMeans

minibatch_kmeans = MiniBatchKMeans(n_clusters=5, random_state=42)
minibatch_kmeans.fit(x)

C:\Users\gulsh\AppData\Roaming\Python\Python312\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 2048 or by setting the environment variable OMP_NUM_THREADS=4
  warnings.warn(


MiniBatchKMeans(n_clusters=5, random_state=42)

if the dataset does not fit properly the best option is to use the memmap class as we did for incremental pca in chap 8 
or you can pass one mini batch at a time to the partial_fit method but this is more tedious as u will need to perform multiple initializations and select the best one urself 

even if the mini k means is much faster than the normal k means as seen in fig 9.6 right almost 3.5 times, the inertia for mini k means is slightly worst as seen in the left of the fig


finding the optimal no of clusters:

till now we have set the no of clusters k = 5 as it was visibly correct, but it wont be easy everytime and if we set it anything the results would be bad seen in fig 9.7

we might think we can pick the bst model by seeing the inertia value, but the model with k =3 has inertia 653.2 and the model with k = 6 has inertia 119.1 musch smaller than the optimal k = 5 which was 211.6, this tell us as the k increases inertia will decrease as its common that the dist between each clusters centeres and instances decreases as we increase the cluster centres

one technique is if we map the inertia as a function of k seen in fig 9.8 we can see theres an point called elbow at k =4, we can say after 4 the inertia decreases very slowly so its the most optimal choice as highrer values from this will ony divide the clusters in half.

another way which is computationally expensive is using the mean silhouette coefficient over all the instances, an instances silh coeff is equal to 
b-a/max(a,b) where 
b is the mean of distances between the instance and the instances of the next closest cluster and 
a is the mean distances between the instance and the instances of its own cluster
the coeff val can vary between 1 and -1, 
the coeff val near 1 means its well inside its own cluster 
near 0 means its in the boundary
and near -1 means its assigned to a wrong cluster 
to compute the silhoutte score we can use the scikit learn

In [15]:
from sklearn.metrics import silhouette_score
silhouette_score(x, kmeans.labels_)

0.6183251804193935

if we map the silh scores for each no of possible clusters in fig 9.9, we can see k =4 is best option but k=5 is still good to go much better than k=6 or 7 and we can see k=3 performed worst as proven in previous graphs 

in fig 9,10 we drew a silh diag by mapping each instances silh score 
we made it for diff k val, the height represents the no of points in that cluster and the width rep silh score, red dash line is average silh score
if many clusters are left of that line the cluster is bad, as it indicates poor seperation 

k= 3 : shapes uneven many points dont cross the red line, clusters are too merged so underclustering

k = 4: most points extend well to the right, high silh score but one cluster is very large so good seperation but imbalanced cluster sizes

k= 5: perfect balancing 

k= 6 too many clusters 


limits of kmeans:

k means is fast and scalable but not perfect as we need to run it multiple times to avoid the suboptimal results and we have to mention the no of clusters, it also struggles when the clusters have varying sizes, different densities, eg : fig9.11 shows k means for 3 ellipsoidal clusters of diff dim and densities and orientations 
on these type of elliptical cluster gaussian models works best
scaling always helps in clustering algos, it does not guarantee that all the clusters will be nice and spherical but it helps the algortihm



clustering for image segmentation 

color segmentation : similar colors pixels gets assigned to the same segment, eg: analyis of satellite images to measure the forest area 

semantic segmentation : pixels part of the same object type gets assigned to the same segment, used in seelf driving cars to assign pixels to a similar segment like "pedestrian" eg: scene understanding

instance segmentation : all pixls belonging to an instance or object is assigned to that same object, like each pedestrian are differnt entities here 

the state of the art semantic or instance seg is achieved using complex arch based on cnn , for now we will focus on color seg using k means 

In [16]:
from PIL import Image
import numpy as np

image = np.asarray(Image.open('image.png')) #converts it to a numpy array
print(image.shape)

(98, 152, 4)


In [17]:
image = image[:, :, :3]
image.shape

(98, 152, 3)

the imageis represented as a 3D array, 
the first dim is height 
the second is width
the third is no of color channels 

we removed the 4th channel which was alpha transparency as it can affect the k means

so for each pixel we will have a 3d vector containing intensities of red green blue assigned between 0 and 255
some images may have fever channels like grayscale which only have one 

In [18]:
X = image.reshape(-1, 3) #flattens the image (98,152,3)→(14896,3)
kmeans = KMeans(n_clusters=8, random_state=42).fit(X)
segmented_img = kmeans.cluster_centers_[kmeans.labels_]
segmented_img = segmented_img.reshape(image.shape)

first we flattens the image meaning each row = one pixel and each pixel = r g b

then we apply kmeans, which groups them in 8 clusters, each cluster = group of similar clusters 

output : kmeans.cluster_centers -> 8 colors centroids 
kmeans.labels_ -> cluster index for each pixel

now each pixel will be changed to its cluster center
if :
kmeans.labels_ = [1, 1, 0]
kmeans.cluster_centers_ = [
    [255, 0, 0],   # cluster 0
    [0, 255, 0]    # cluster 1
]
then:
segmented_img =
[
  [0, 255, 0],
  [0, 255, 0],
  [255, 0, 0]
]

then reshape it back to image 

the output is shown in the upper right of the fig 9-12
we can experiment using many differnet no of clusters showin in fog 9-12 but as we go lower than 8 the ladybirds red color did not remain a seperate cluster anymore as k means prefers clusters of similar sizes, the ladybird color is much smaller than the others, so k means fails to detect it 



USING CLUSTERING FOR SEMI SUPERVISED LEARNING 

we can also use clustering in semi supervised learning, when we have plenty of unlabeled dataset and a very few labeled ones, we will be using mnist for this 
load and split the dataset :

In [19]:
from sklearn.datasets import load_digits
X_digits, y_digits = load_digits(return_X_y=True)
X_train, y_train = X_digits[:1400], y_digits[:1400]
X_test, y_test = X_digits[1400:], y_digits[1400:]

for now we will assume we only have 50 labeled instances, to get a baseline performance lets train a logistic regression model on these 50 labeled instances 

we can then check the performance of this model on the test set 

In [20]:
from sklearn.linear_model import LogisticRegression
n_labeled = 50
log_reg = LogisticRegression(max_iter=10_000)
log_reg.fit(X_train[:n_labeled], y_train[:n_labeled])

LogisticRegression(max_iter=10000)

In [21]:
log_reg.score(X_test, y_test)

0.7581863979848866

the models accuracy is 74.8, not good but if we train on te full dataset it will reach 90, lets see how we improve this 

first we define 50 clusters in the training set, then for each cluster we will find the image closest to the centroid  see fig 9-13 for rep

In [22]:
k = 50
kmeans = KMeans(n_clusters=k, random_state=42)
X_digits_dist = kmeans.fit_transform(X_train)
representative_digit_idx = np.argmin(X_digits_dist, axis=0)
X_representative_digits = X_train[representative_digit_idx]

C:\Users\gulsh\AppData\Roaming\Python\Python312\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


In [25]:
y_representative_digits = np.array([1, 3, 6, 0, 7, 9, 2,4,8,9,5,4,7,4,2,6,7,2,5,1,4,1,3,3,8,8,2,5,6,9,1,4,0,6,8,3,4,6,7,2,4,1,0,7, 5, 1, 9, 9, 3, 7])

In [26]:
log_reg = LogisticRegression(max_iter=10_000)
log_reg.fit(X_representative_digits, y_representative_digits)
log_reg.score(X_test, y_test)

0.14609571788413098

the accuracy will increase we didnt label it correctly here, instead of ramdomly label them we should label the representive instances 

but we can go one step ahead by propogating the labels to all the other instances in the same cluster this is called labelled propogation , the performance will increase further 

lets move one step further 
if we ignore the 1% instances that are farthest from their closest center, this should eliminate some outliers, the accuracy will increase further almost the same what we got with the full dataset 

there are two other classes LabelSpreading and LabelPropofation , both classes construct similarity matrices for all the instances and then iteratively propogate the labeled instances to the unlabeled instances 

there is one more class SelfTrainingClassifier which uses a base classifier such as random forest classifier and trains on labeled instances, then uses it to predict labels for unlabeled samples , it then updates the training set with labels it is most confident about and repeats this process of training and labelling until it cannot add labels anymore it can give model a little boost 

ACTIVE LEARNING

to continue improving your model we use few rounds of active learning where a human interact with the learning algo, where the algo reqs the expert to provide the labels for some specific instances , there are diff strategies for active learning but one of the most common is uncertainity sampling told in the para 

other active learning strat include:
labelling the instances that would result in the largest model change, or largest drop in the models validation error or the instances that diff models disagree on

before we study about gaussian mixture models, lets study DBSCAN which works on local density estimation, this helps in identifying the clusters of arbitary shapes or random shapes 